# 01 · 300W + FAN pipeline walkthrough

Visual sanity check of the `300w_fan` experiment config, step by step, on a few random images:

1. **Dataset** — 300W samples with their 68 annotated ibug points (GT).
2. **Detection** — YuNet face detector (boxes, scores, 5 keypoints).
3. **Target selection** — `gt_match`: pick the detection whose box best overlaps the GT points' box.
4. **Face crop** — the landmark model's own square crop around the selected detection (exactly what FAN receives).
5. **Landmark model** — `face_alignment` (2D-FAN): predictions vs GT, per-image NME and mean signed offset.

Change `N_SAMPLES` / `SEED` / `INDICES` in the *Pick samples* cell to look at other images.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from facemeasure import (
    FaceDataset,
    SelectionStatus,
    build_detector,
    build_landmark_model,
    build_selector,
    load_config,
)
from facemeasure.config import PROJECT_ROOT
from facemeasure.geometry import apply_affine, points_to_box, union_box
from facemeasure.viz import (
    C_CROP,
    C_GT_BOX,
    C_GT_PTS,
    C_KPTS,
    C_OTHER,
    C_PRED,
    C_REJECTED,
    C_SELECTED,
    C_TOP,
    OUTLINE,
    crop_outline,
    draw_box,
    draw_detection,
    draw_ibug68,
    legend_below,
    region_handles,
    region_legend,
    show,
    zoom_to,
)

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})
%config InlineBackend.figure_formats = ["jpeg"]  # photo-heavy figures: far smaller notebook than png
print("project root:", PROJECT_ROOT)

In [ ]:
cfg = load_config("300w_fan")
print(cfg.model_dump_json(indent=2))

## 1. Dataset

In [ ]:
ds = FaceDataset(cfg.data.paths)
schemas = ds.table["schema"].unique().to_pylist()
print(f"{len(ds)} samples, point schemas: {schemas}")

### Pick samples
`SEED = None` gives a new random draw every run; the chosen indices are printed so a draw can be reproduced via `INDICES`.

In [ ]:
N_SAMPLES = 4
SEED = None     # int -> reproducible draw
INDICES = [252, 358, 513, 101]  # e.g. [12, 345, 7, 501] to inspect specific images (overrides the random draw)

if INDICES is None:
    seed = SEED if SEED is not None else int(np.random.SeedSequence().entropy % 2**32)
    INDICES = np.random.default_rng(seed).choice(len(ds), size=N_SAMPLES, replace=False).tolist()
    print(f"seed = {seed}")
print(f"INDICES = {INDICES}")

samples = [ds[i] for i in INDICES]
for i, s in zip(INDICES, samples):
    h, w = s.image.shape[:2]
    print(f"[{i:>3}] {s.image_id:<16} {w}x{h}  points={s.points.shape}  schema={s.schema}")

### GT points (as stored in the parquet)
Top row: full image with the 68 points and their bounding box (the box `gt_match` matches against).
Bottom row: zoom on the annotated face.

In [ ]:
fig, axes = plt.subplots(2, len(samples), figsize=(4.2 * len(samples), 8.5), squeeze=False,
                         layout="compressed")
for col, (idx, s) in enumerate(zip(INDICES, samples)):
    gt_box = points_to_box(s.points)
    for row in range(2):
        ax = axes[row, col]
        show(ax, s.image, f"[{idx}] {s.image_id}" if row == 0 else "zoom on annotated face")
        draw_ibug68(ax, s.points, s=6 if row == 0 else 14, lines=row == 1)
        draw_box(ax, gt_box, C_GT_BOX, lw=1.2, ls="--")
        if row == 1:
            zoom_to(ax, gt_box, s.image.shape)
region_legend(fig)
plt.show()

### Point index layout
Point indices on the first sample, to check the dataset's ordering matches the ibug68 convention.

In [ ]:
s = samples[0]
fig, ax = plt.subplots(figsize=(9, 9), layout="compressed")
show(ax, s.image, f"{s.image_id}: ibug68 indices")
draw_ibug68(ax, s.points, s=18, labels=True)
zoom_to(ax, points_to_box(s.points), s.image.shape, margin=0.1)
region_legend(fig)
plt.show()

## 2. Detection — YuNet
All detections on the full image, using the config's `score_threshold` and `max_side` (large images are
downscaled before detection; outputs are mapped back to original pixels).

In [ ]:
detector = build_detector(cfg.detector)
print(f"{type(detector).__name__}: params={cfg.detector.params}, keypoints={detector.keypoint_names}")

detections = []
for idx, s in zip(INDICES, samples):
    t0 = time.perf_counter()
    dets = detector(s.image)
    dt = (time.perf_counter() - t0) * 1000
    detections.append(dets)
    h, w = s.image.shape[:2]
    scale = min(1.0, detector.max_side / max(h, w)) if detector.max_side else 1.0
    scores = ", ".join(f"{d.score:.2f}" for d in dets[:5]) + (" ..." if len(dets) > 5 else "")
    print(f"[{idx:>3}] {w}x{h} (det scale {scale:.2f}) {dt:6.1f} ms  "
          f"{len(dets):>3} detections  scores: {scores or '-'}")

In [ ]:
MAX_LABELS = 15  # don't print scores on every box in crowd photos

fig, axes = plt.subplots(1, len(samples), figsize=(4.2 * len(samples), 4.2), squeeze=False,
                         layout="compressed")
for ax, idx, s, dets in zip(axes[0], INDICES, samples, detections):
    show(ax, s.image, f"[{idx}] {len(dets)} detection(s)")
    few = len(dets) <= MAX_LABELS
    for d in reversed(dets):  # draw best on top
        draw_detection(ax, d, C_TOP if d is dets[0] else C_OTHER, lw=1.5 if few else 0.8,
                       label=f"{d.score:.2f}" if few else None, kpts=few)
legend_below(fig, [
    Line2D([], [], color=C_TOP, lw=2, label="highest score"),
    Line2D([], [], color=C_OTHER, lw=2, label="other detections"),
    Line2D([], [], color=C_KPTS, marker="x", ls="", label=f"YuNet keypoints (if ≤ {MAX_LABELS} faces)"),
])
plt.show()

## 3. Target selection — `gt_match`
Each detection box is compared with the GT box (min/max of the 68 points); the best IoU wins if it is
≥ `iou_threshold`. Statuses: `ok`, `no_match` (best IoU too low), `no_detection`.

Left: full image (best candidate colored by outcome, others gray). Right: zoom on GT box + best candidate.

In [ ]:
selector = build_selector(cfg.target_selector)
print(f"{type(selector).__name__}: iou_threshold={selector.iou_threshold}")

selections = [selector(dets, gt_points=s.points) for s, dets in zip(samples, detections)]
for idx, sel in zip(INDICES, selections):
    iou = f"{sel.best_iou:.3f}" if sel.best_iou is not None else "-"
    score = f"{sel.detection.score:.2f}" if sel.detection is not None else "-"
    print(f"[{idx:>3}] status={sel.status:<12} n_det={sel.n_detections:<4} "
          f"best_index={sel.best_index}  best_iou={iou}  selected_score={score}")

In [ ]:
fig, axes = plt.subplots(len(samples), 2, figsize=(11, 3.9 * len(samples)), squeeze=False,
                         gridspec_kw={"width_ratios": [1.4, 1]}, layout="compressed")
for (ax_full, ax_zoom), idx, s, dets, sel in zip(axes, INDICES, samples, detections, selections):
    best = dets[sel.best_index] if sel.best_index is not None else None
    best_color = C_SELECTED if sel.status == SelectionStatus.OK else C_REJECTED
    iou_txt = f", IoU={sel.best_iou:.2f}" if sel.best_iou is not None else ""
    for ax, zoomed in ((ax_full, False), (ax_zoom, True)):
        show(ax, s.image, f"[{idx}] {s.image_id}: {sel.status}{iou_txt}" if not zoomed else
             f"zoom: matched candidate #{sel.best_index}" if sel.status == SelectionStatus.OK else "zoom on GT box")
        for i, d in enumerate(dets):
            if i != sel.best_index:
                draw_box(ax, d.box, C_OTHER, lw=0.8)
        draw_box(ax, sel.target_box, C_GT_BOX, lw=1.5, ls="--",
                 label="GT box" if zoomed else None, label_pos="bottom")
        if best is not None:
            draw_detection(ax, best, best_color, lw=2,
                           # rejected candidates are outside the zoom, so label them on the full image
                           label=f"#{sel.best_index} IoU {sel.best_iou:.2f} · score {best.score:.2f}"
                           if zoomed == (sel.status == SelectionStatus.OK) else None,
                           kpts=zoomed)
        if zoomed:
            draw_ibug68(ax, s.points, s=8, lines=False)
            # include the candidate only if it matched; a rejected one may be far away (it's visible on the left)
            ok_box = best.box if sel.status == SelectionStatus.OK else None
            zoom_to(ax, union_box(sel.target_box, ok_box), s.image.shape, margin=0.25)
legend_below(fig, [
    Line2D([], [], color=C_GT_BOX, ls="--", lw=2, label="GT box (from points)",
           path_effects=OUTLINE),
    Line2D([], [], color=C_SELECTED, lw=2, label=f"selected (IoU ≥ {selector.iou_threshold})"),
    Line2D([], [], color=C_REJECTED, lw=2, label="best candidate, rejected"),
    Line2D([], [], color=C_OTHER, lw=2, label="other detections"),
])
plt.show()

## 4. Face crop — what FAN receives


Left: source image with the selected detection box and the crop square (mapped back through `inverse`).
Right: the crop, with the GT points and the detection box mapped into it through `matrix`; black = outside the
source image. The points must land on the face features; if they do, `matrix`/`inverse` are correct and are
what maps FAN's predictions back to source pixels.

In [ ]:
landmark_model = build_landmark_model(cfg.landmark_model)
size = landmark_model.crop_size
crops = [landmark_model.crop(s.image, sel.detection) if sel.status == SelectionStatus.OK else None
         for s, sel in zip(samples, selections)]

fig, axes = plt.subplots(len(samples), 2, figsize=(9, 4.3 * len(samples)), squeeze=False, layout="compressed")
for (ax_src, ax_crop), idx, s, sel, crop in zip(axes, INDICES, samples, selections, crops):
    if crop is None:
        show(ax_src, s.image, f"[{idx}] {s.image_id}: {sel.status}")
        ax_crop.set_axis_off()
        ax_crop.text(0.5, 0.5, f"{sel.status}\nno crop", ha="center", va="center", transform=ax_crop.transAxes)
        continue
    outline = crop_outline(crop)

    show(ax_src, s.image, f"[{idx}] {s.image_id}: source")
    draw_box(ax_src, sel.detection.box, C_SELECTED, lw=1.5)
    ax_src.plot(*outline.T, color=C_CROP, lw=2, path_effects=OUTLINE)
    draw_ibug68(ax_src, s.points, s=4, lines=False)
    # not zoom_to: it clips to the image, and the square may extend past the border
    x1, y1, x2, y2 = points_to_box(outline)
    m = 0.05 * (x2 - x1)
    ax_src.set_xlim(x1 - m, x2 + m)
    ax_src.set_ylim(y2 + m, y1 - m)

    show(ax_crop, crop.image, f"crop {size}x{size} (x{crop.matrix[0, 0]:.2f})")
    det_in_crop = apply_affine(crop.matrix, sel.detection.box.reshape(2, 2)).ravel()
    draw_box(ax_crop, det_in_crop, C_SELECTED, lw=1.5)
    draw_ibug68(ax_crop, apply_affine(crop.matrix, s.points), s=10)
    ax_crop.set_xlim(-0.5, size - 0.5)
    ax_crop.set_ylim(size - 0.5, -0.5)
legend_below(fig, [
    Line2D([], [], color=C_SELECTED, lw=2, label="selected detection"),
    Line2D([], [], color=C_CROP, lw=2, label="crop square", path_effects=OUTLINE),
    *region_handles(),
])
plt.show()

## 5. Landmark model — FAN
`landmark_model` (built in section 4) runs on the selected detection of each sample: the crop above → 2D-FAN
heatmaps (with `flip_test`, averaged with the mirrored crop) → points mapped back to source pixels.

Per image: **NME** = mean point-to-point error / inter-ocular distance (outer eye corners, points 36 and 45),
and the **mean signed offset** (pred − GT) in pixels, which shows a systematic shift that NME hides.
GT is used as stored in the parquet: 0-based pixel centers (`prepare_300w.py` subtracts 1 px from 300W's
1-based `.pts`, see `docs/decisions.md`).

In [ ]:
EYE_OUTER = (36, 45)  # ibug68 outer eye corners (subject's right, left)

preds, nmes, offsets = [], [], []
for idx, s, sel, crop in zip(INDICES, samples, selections, crops):
    if sel.status != SelectionStatus.OK:
        preds.append(None)
        print(f"[{idx:>3}] {sel.status}: no prediction")
        continue
    t0 = time.perf_counter()
    pred = landmark_model(s.image, sel.detection)
    dt = (time.perf_counter() - t0) * 1000
    preds.append(pred)
    assert np.array_equal(pred.crop.image, crop.image), "model ran on a different crop than section 4 shows"

    err = pred.points.astype(np.float64) - s.points       # (68, 2) pred - GT, px
    iod = np.linalg.norm(s.points[EYE_OUTER[0]] - s.points[EYE_OUTER[1]])
    nme = np.linalg.norm(err, axis=1).mean() / iod
    nmes.append(nme)
    offsets.append(err.mean(0))
    print(f"[{idx:>3}] NME {nme * 100:5.2f}% (IOD {iod:5.1f}px)  mean offset pred-GT "
          f"dx {err[:, 0].mean():+6.2f} dy {err[:, 1].mean():+6.2f} px  "
          f"score median {np.median(pred.scores):.2f} min {pred.scores.min():.2f}  {dt:6.0f} ms")
if nmes:
    dx, dy = np.mean(offsets, axis=0)
    print(f"mean over {len(nmes)} image(s): NME {np.mean(nmes) * 100:.2f}%  offset dx {dx:+.2f} dy {dy:+.2f} px")

Left: full image, GT (white) and prediction (red). Right: zoom on the face, with contours and a thin line
from each GT point to its prediction.

In [ ]:
fig, axes = plt.subplots(len(samples), 2, figsize=(11, 3.9 * len(samples)), squeeze=False,
                         gridspec_kw={"width_ratios": [1.4, 1]}, layout="compressed")
nme_by_sample = iter(nmes)
for (ax_full, ax_zoom), idx, s, sel, pred in zip(axes, INDICES, samples, selections, preds):
    if pred is None:
        show(ax_full, s.image, f"[{idx}] {s.image_id}: {sel.status}")
        ax_zoom.set_axis_off()
        ax_zoom.text(0.5, 0.5, f"{sel.status}\nno prediction", ha="center", va="center", transform=ax_zoom.transAxes)
        continue
    nme = next(nme_by_sample)
    show(ax_full, s.image, f"[{idx}] {s.image_id}: NME {nme * 100:.2f}%")
    draw_ibug68(ax_full, s.points, s=5, lines=False, color=C_GT_PTS)
    draw_ibug68(ax_full, pred.points, s=5, lines=False, color=C_PRED)

    show(ax_zoom, s.image, "zoom on the face")
    for g, p in zip(s.points, pred.points):
        ax_zoom.plot([g[0], p[0]], [g[1], p[1]], color="yellow", lw=0.8, zorder=2)
    draw_ibug68(ax_zoom, s.points, s=10, color=C_GT_PTS)
    draw_ibug68(ax_zoom, pred.points, s=10, color=C_PRED)
    zoom_to(ax_zoom, union_box(points_to_box(s.points), points_to_box(pred.points)), s.image.shape, margin=0.1)
legend_below(fig, [
    Line2D([], [], color=C_GT_PTS, marker="o", markeredgecolor="black", lw=1, label="GT"),
    Line2D([], [], color=C_PRED, marker="o", markeredgecolor="black", lw=1, label="FAN prediction"),
    Line2D([], [], color="yellow", lw=1, label="GT → prediction"),
])
plt.show()